# 🎬 MiniMax H3 Video Generation on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keboqi/minimax-h3/blob/main/minimax_h3_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue.svg)](https://github.com/keboqi/minimax-h3)

A streamlined Google Colab deployment notebook for **MiniMax H3**, featuring:
- **Instant Startup**: No heavy model pre-downloads at startup. All model weights download on demand when you first use them.
- **Google Drive Persistence**: Optional 1-click mounting of Google Drive for ComfyUI and Gradio outputs so all generated videos and images survive across Colab instances.
- **Unified Media Generation**: Text-to-Video, First/Last-frame conditioning, Reference-media conditioning, FL2VA voice references, and native audio soundtrack decoding.
- **Dedicated Creative Tabs**: Qwen Image 2.1 (T2I & edit), LTX-2.5 (T2V & I2V with synced audio), MiniMax Music 3, and YuE2 song generation.
- **Hardware Acceleration**: SLA (Block Sparse Attention), Sol-Attn, Comfy Kitchen attention, SageAttention 2, and Spectrum forecasting.
- **Model Profiles**: Singularity pruned v1.3 INT8, Speed NVFP4, Quality INT8 ConvRot, and Original BF16.
- **High-Speed Turbo Modes**: LightX2V 4-step/8-step, TaoMate 3-step, and Larry v4-600 EMA LoRAs.
- **Proxied ComfyUI**: Integrated ComfyUI workflow editor accessible directly through `/comfyui`.

---
### 💡 Hardware & Runtime Recommendations
- **Recommended**: **NVIDIA A100** (40GB/80GB) or **NVIDIA L4** (24GB) via Colab Pro / Pro+.
- **NVIDIA T4** (16GB, Free tier): Supported, but VRAM is tight. Please ensure **"Offload models between H3 stages"** remains enabled in the UI and use the **Singularity** or **NVFP4 / Fast** presets.
- **GPU Setup**: Make sure a GPU runtime is selected (**Runtime → Change runtime type → T4/L4/A100 GPU**).


In [ ]:
# @title 1. Verify GPU Hardware and System Resources
import os
import subprocess

print("Checking NVIDIA GPU hardware...")
!nvidia-smi

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"\n[OK] Detected GPU: {gpu_info}")
    if "T4" in gpu_info:
        print("[i] Running on NVIDIA T4 (16GB VRAM).")
        print("    Tip: Keep 'Offload models between H3 stages' enabled in the UI for optimal memory usage.")
    elif any(k in gpu_info for k in ["A100", "H100", "L4"]):
        print("[OK] High-performance GPU detected with ample VRAM for MiniMax H3!")
except Exception as exc:
    print(f"[!] Warning checking GPU: {exc}")


### 2. Clone Repository & Setup Python 3.12 Environment
MiniMax H3 requires Python 3.12, PyTorch 2.11 + CUDA 13.0, and prebuilt acceleration wheels (such as SageAttention 2.2.0 cp312).
We use `uv` to automatically provision a standalone Python 3.12 virtual environment in seconds.


In [ ]:
# @title 2. Clone Repository & Create Python 3.12 Virtualenv
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/keboqi/minimax-h3.git"  # @param {type:"string"}
WORKSPACE_DIR = "/content/minimax-h3"
VENV_DIR = "/content/venv_h3"

# 1. Clone repository
if not (Path(WORKSPACE_DIR) / "run_h3.sh").is_file():
    print(f"Cloning MiniMax H3 from {REPO_URL}...")
    !git clone {REPO_URL} {WORKSPACE_DIR}
else:
    print(f"[OK] Repository already present at {WORKSPACE_DIR}; pulling latest updates...")
    !git -C {WORKSPACE_DIR} remote set-url origin {REPO_URL} 2>/dev/null || true
    !git -C {WORKSPACE_DIR} fetch origin HEAD
    !git -C {WORKSPACE_DIR} reset --hard FETCH_HEAD

%cd {WORKSPACE_DIR}

# 2. Install uv and provision Python 3.12 environment
print("\nSetting up isolated Python 3.12 virtual environment with uv...")
!pip install --quiet "uv>=0.8"
!uv venv --python 3.12 {VENV_DIR}

# 3. Add venv to PATH so all subshells and scripts use Python 3.12
os.environ["PATH"] = f"{VENV_DIR}/bin:{os.environ['PATH']}"
os.environ["VIRTUAL_ENV"] = VENV_DIR

print("\nVerifying active Python version:")
!python3 --version
!which python3


### 3. Configure Hugging Face Token (Recommended)
An authentication token allows downloading gated models (e.g., LTX-2.5, Gemma) and avoids Hugging Face Hub download rate limits on Colab.
- You can create a free token with Read permissions at: https://huggingface.co/settings/tokens
- You can either paste it below or add `HF_TOKEN` in Colab's **Secrets** menu (the key icon in the left toolbar).


In [ ]:
# @title 3. Configure Hugging Face Token
import os

HF_TOKEN = ""  # @param {type:"string"}

# Try reading from Colab Secrets if not specified above
if not HF_TOKEN.strip():
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ""
    except Exception:
        pass

if not HF_TOKEN.strip():
    HF_TOKEN = os.getenv("HF_TOKEN", "")

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("[OK] Hugging Face token configured.")
else:
    print("[!] No Hugging Face token specified. Default models will download unauthenticated.")
    print("    Note: Gated models (like LTX-2.5) require an authenticated token.")


### 4. (Optional) Mount Google Drive for Persistent Outputs
By default, files created in Google Colab are temporary and deleted when your session terminates or disconnects.
Enable this option to link ComfyUI and Gradio output folders to your **Google Drive** (`MyDrive/MiniMax-H3/`):
- **Survive Across Instances**: All generated videos, images, and audio remain in your Google Drive forever.
- **Gallery Persistence**: When you start a new Colab runtime in the future, the Gallery will automatically recognize and display all previously generated assets.


In [ ]:
# @title 4. (Optional) Mount Google Drive for Persistent Outputs
# @markdown Link ComfyUI and Gradio output directories to Google Drive so generated assets persist across sessions.

MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MiniMax-H3"  # @param {type:"string"}

import os
import shutil
from pathlib import Path

def setup_google_drive_outputs(enabled: bool = True, drive_folder: str = "/content/drive/MyDrive/MiniMax-H3"):
    if not enabled:
        print("[i] Google Drive persistence disabled. Outputs will be saved locally in this Colab session.")
        return

    try:
        from google.colab import drive
        print("Mounting Google Drive at /content/drive...")
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"[!] Could not mount Google Drive: {exc}")
        print("    Continuing with local runtime output directory.")
        return

    gdrive_base = Path(drive_folder).expanduser().resolve()
    gdrive_comfy_output = gdrive_base / "output"
    gdrive_gradio_output = gdrive_base / "gradio_outputs"

    gdrive_comfy_output.mkdir(parents=True, exist_ok=True)
    gdrive_gradio_output.mkdir(parents=True, exist_ok=True)

    workspace = Path("/content/minimax-h3")
    comfy_output = workspace / "h3" / "ComfyUI" / "output"
    gradio_output = workspace / "h3" / "gradio_outputs"

    comfy_output.parent.mkdir(parents=True, exist_ok=True)
    gradio_output.parent.mkdir(parents=True, exist_ok=True)

    for local_path, gdrive_target in [
        (comfy_output, gdrive_comfy_output),
        (gradio_output, gdrive_gradio_output),
    ]:
        if local_path.is_symlink():
            if local_path.resolve() == gdrive_target:
                continue
            local_path.unlink()
        elif local_path.is_dir():
            for item in local_path.iterdir():
                target_item = gdrive_target / item.name
                if not target_item.exists():
                    shutil.move(str(item), str(target_item))
            shutil.rmtree(local_path)
        elif local_path.exists():
            local_path.unlink()

        try:
            local_path.symlink_to(gdrive_target, target_is_directory=True)
        except OSError:
            os.system(f"ln -sfn '{gdrive_target}' '{local_path}'")

    print("[OK] Google Drive output folders linked successfully!")
    print(f"     ComfyUI output  -> {gdrive_comfy_output}")
    print(f"     Gradio outputs  -> {gdrive_gradio_output}")
    print("     All generated assets will persist in your Google Drive across sessions.")

setup_google_drive_outputs(MOUNT_GOOGLE_DRIVE, DRIVE_OUTPUT_DIR)


### 5. Launch MiniMax H3 (`bash run_h3.sh`)
You can launch the app directly using the cell below, or by running `bash run_h3.sh` in the Colab Terminal.

```bash
cd /content/minimax-h3
bash run_h3.sh
```

- **Zero Wait at Startup**: Models are **not** preloaded at startup. The servers start in seconds!
- **On-Demand Downloads**: Checkpoints, text encoders, VAEs, and LoRAs download automatically on first use when you click **Generate**.
- **Public URLs**: A public `*.gradio.live` link and an optional Cloudflare `*.trycloudflare.com` tunnel URL are generated.


In [ ]:
# @title 5. Launch MiniMax H3 (Gradio & ComfyUI)
import os
import time
import subprocess
import threading
import re

%cd /content/minimax-h3

ENABLE_CLOUDFLARE_TUNNEL = True  # @param {type:"boolean"}

# Re-verify Google Drive output links if enabled
if globals().get("MOUNT_GOOGLE_DRIVE", False) and "setup_google_drive_outputs" in globals():
    setup_google_drive_outputs(MOUNT_GOOGLE_DRIVE, globals().get("DRIVE_OUTPUT_DIR", "/content/drive/MyDrive/MiniMax-H3"))

def launch_cloudflare():
    """Background helper to launch cloudflared tunnel for port 7860."""
    try:
        if not os.path.exists("/usr/local/bin/cloudflared"):
            print("[Cloudflare] Downloading cloudflared binary...")
            subprocess.run(
                ["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"],
                check=True
            )
            os.chmod("/usr/local/bin/cloudflared", 0o755)

        time.sleep(3)  # Start tunnel early so public URL is ready quickly
        proc = subprocess.Popen(
            ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        for line in proc.stderr:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                url = match.group(0)
                print("\n" + "=" * 62)
                print(f"🚀 Cloudflare Tunnel Public URL: {url}")
                print(f"🎨 ComfyUI Editor Proxy:        {url}/comfyui")
                print("=" * 62 + "\n", flush=True)
                break
    except Exception as exc:
        print(f"[Cloudflare Tunnel] Info: {exc}")

if ENABLE_CLOUDFLARE_TUNNEL:
    threading.Thread(target=launch_cloudflare, daemon=True).start()

# Launch via run_h3.sh (starts ComfyUI + Gradio, creates Gradio live share tunnel)
# Ensure entrypoint exists
if not (Path("/content/minimax-h3/gradio_app.py")).is_file():
    !git -C /content/minimax-h3 checkout HEAD -- gradio_app.py 2>/dev/null || true

!bash run_h3.sh


## 💡 Tips & Operation Guide

### 🎛️ User Interface & ComfyUI Access
- **Gradio Interface**: Generates video, image frames, and audio soundtracks with real-time preview and multi-stage progress reporting.
- **ComfyUI Editor**: Open `<public-url>/comfyui` in your browser (or use the ComfyUI tab in the interface) to view and edit the active node graphs live.

### 💾 Persistent Outputs on Google Drive
- When **Mount Google Drive** is enabled, all assets are stored under `MyDrive/MiniMax-H3/`.
- **ComfyUI Outputs**: Direct node saves (images, audio, videos) go to `MyDrive/MiniMax-H3/output`.
- **Gradio Outputs**: Upscaled videos, generated videos, audio tracks, and gallery items go to `MyDrive/MiniMax-H3/gradio_outputs`.
- When reconnecting in future sessions, all previous files are immediately accessible in your Google Drive and appear in the UI gallery.

### ⚡ Performance & VRAM Tips
- **VRAM Offload**: If running on GPUs with 16GB–24GB VRAM (such as T4 or L4), ensure **"Offload models between H3 stages"** is enabled in the model settings.
- **Attention Backend**: Default **SLA Fast** uses block-sparse attention and dense final step for fast audio-safe generation. You can also select **Comfy Kitchen** or **SageAttention 2**.
- **Turbo Modes**:
  - **LightX2V Turbo**: 4-step or 8-step LoRA for ultra-fast previews.
  - **Larry v4-600 EMA**: 6-step quality-focused Turbo LoRA.
  - **TaoMate 3-step**: 3-step distilled generation.
- **Stop Server**: Click the Stop button on the Colab execution cell to cleanly terminate ComfyUI and Gradio.
